# 🚗💨 CO2 Emission by Vehicles — Prediction System
## Notebook 1: Quick Demo (100 Instances, 3-Fold Cross-Validation)

---
**Course:** AIC354 — Machine Learning Fundamentals  
**Assignment:** 3 — Applying, Evaluating & Comparing Multiple ML Algorithms  
**Dataset:** CO2 Emissions Canada (Government of Canada Open Data)  

---
### 🤖 Algorithms Used (Different from Car Price System)
| # | Algorithm | Type |
|---|-----------|------|
| 1 | Ridge Regression | Linear |
| 2 | Lasso Regression | Linear |
| 3 | Support Vector Regression (SVR) | Kernel-based |
| 4 | Extra Trees Regressor | Ensemble |
| 5 | XGBoost Regressor | Boosting |

> **Evaluation Metric:** RMSE (Root Mean Squared Error) — lower is better  
> **Cross-Validation:** 3-Fold on 100 randomly sampled instances


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install xgboost tqdm --quiet

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import time
import sys

from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
from xgboost import XGBRegressor

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error

from google.colab import files
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('✅ All libraries imported successfully!')

In [ ]:
# ── Upload Dataset ────────────────────────────────────────────────────────────
print('📂 Please upload the dataset file: CO2 Emissions_Canada.csv')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_full = pd.read_csv(filename)
print(f'\n✅ Dataset loaded: {df_full.shape[0]:,} rows × {df_full.shape[1]} columns')

In [ ]:
# ── Exploratory Data Analysis ─────────────────────────────────────────────────
print('=' * 60)
print('  DATASET OVERVIEW')
print('=' * 60)
print(df_full.head())
print('\nShape:', df_full.shape)
print('\nData Types:')
print(df_full.dtypes)
print('\nMissing Values:')
print(df_full.isnull().sum())
print('\nBasic Statistics:')
print(df_full.describe())

In [ ]:
# ── Sample 100 Instances ──────────────────────────────────────────────────────
df = df_full.sample(n=100, random_state=42).reset_index(drop=True)
print(f'✅ Sampled 100 instances from {df_full.shape[0]:,} total records.')
print(f'   Target column: CO2 Emissions(g/km)')
print(f'   Target range : {df["CO2 Emissions(g/km)"].min()} – {df["CO2 Emissions(g/km)"].max()} g/km')

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
TARGET = 'CO2 Emissions(g/km)'
DROP_COLS = ['Make', 'Model']  # High-cardinality text, drop for simplicity
CAT_COLS  = ['Vehicle Class', 'Transmission', 'Fuel Type']
NUM_COLS  = ['Engine Size(L)', 'Cylinders',
             'Fuel Consumption City (L/100 km)',
             'Fuel Consumption Hwy (L/100 km)',
             'Fuel Consumption Comb (L/100 km)',
             'Fuel Consumption Comb (mpg)']

df_proc = df.drop(columns=DROP_COLS).copy()

# Label-encode categorical columns
le = LabelEncoder()
for col in CAT_COLS:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

X = df_proc.drop(columns=[TARGET])
y = df_proc[TARGET]

print(f'✅ Feature matrix X: {X.shape}')
print(f'   Features: {list(X.columns)}')

In [ ]:
# ── Define Models with Pipelines ──────────────────────────────────────────────
models = {
    'Ridge Regression'       : Pipeline([('scaler', StandardScaler()),
                                          ('model',  Ridge(alpha=1.0))]),
    'Lasso Regression'       : Pipeline([('scaler', StandardScaler()),
                                          ('model',  Lasso(alpha=0.01, max_iter=10000))]),
    'SVR (RBF Kernel)'       : Pipeline([('scaler', StandardScaler()),
                                          ('model',  SVR(kernel='rbf', C=100, epsilon=0.1))]),
    'Extra Trees Regressor'  : ExtraTreesRegressor(n_estimators=200, random_state=42),
    'XGBoost Regressor'      : XGBRegressor(n_estimators=200, learning_rate=0.05,
                                             max_depth=6, random_state=42,
                                             verbosity=0, eval_metric='rmse'),
}
print(f'✅ {len(models)} models defined.')
for name in models:
    print(f'   • {name}')

In [ ]:
# ── 3-Fold Cross-Validation with Progress Bar ─────────────────────────────────
K_FOLDS = 3
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

results = {}
print(f'\n🔁 Running {K_FOLDS}-Fold Cross-Validation on 100 instances...\n')
print('─' * 55)

for name, model in tqdm(models.items(), desc='Training Models', unit='model'):
    start = time.time()
    neg_mse = cross_val_score(model, X, y, cv=kf,
                              scoring='neg_mean_squared_error',
                              n_jobs=-1)
    rmse_scores = np.sqrt(-neg_mse)
    elapsed = time.time() - start
    results[name] = {
        'RMSE Mean' : round(rmse_scores.mean(), 4),
        'RMSE Std'  : round(rmse_scores.std(),  4),
        'Time (s)'  : round(elapsed, 2),
    }
    print(f'✔ {name:<26} | RMSE: {rmse_scores.mean():.4f} ± {rmse_scores.std():.4f}  ({elapsed:.2f}s)')

print('─' * 55)
print('\n✅ Cross-validation complete!')

In [ ]:
# ── Results Comparison Table ──────────────────────────────────────────────────
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Model', 'RMSE Mean', 'RMSE Std', 'Time (s)']
results_df = results_df.sort_values('RMSE Mean')
results_df['Rank'] = range(1, len(results_df) + 1)
results_df = results_df[['Rank', 'Model', 'RMSE Mean', 'RMSE Std', 'Time (s)']]

print('\n' + '=' * 65)
print('  MODEL COMPARISON — 3-Fold CV on 100 Instances')
print('=' * 65)
print(results_df.to_string(index=False))
print('=' * 65)
best = results_df.iloc[0]
print(f'\n🏆 Best Model: {best["Model"]} (RMSE = {best["RMSE Mean"]})')

In [ ]:
# ── Comparison Charts ─────────────────────────────────────────────────────────
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(results_df))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CO2 Emission Prediction — Model Comparison (3-Fold CV, 100 Instances)',
             fontsize=13, fontweight='bold', y=1.02)

# Bar chart — RMSE Mean
axes[0].barh(results_df['Model'], results_df['RMSE Mean'], color=colors, edgecolor='white')
axes[0].set_xlabel('RMSE Mean (g/km)', fontsize=11)
axes[0].set_title('RMSE Mean by Model', fontsize=12)
axes[0].invert_yaxis()
for i, (val, std) in enumerate(zip(results_df['RMSE Mean'], results_df['RMSE Std'])):
    axes[0].text(val + 0.1, i, f'{val:.2f} ± {std:.2f}', va='center', fontsize=9)
gold_patch = mpatches.Patch(color='#2ecc71', label='Best Model')
axes[0].legend(handles=[gold_patch], loc='lower right')

# Bar chart — Training Time
axes[1].barh(results_df['Model'], results_df['Time (s)'],
             color='#9b59b6', edgecolor='white')
axes[1].set_xlabel('Training Time (seconds)', fontsize=11)
axes[1].set_title('Training Time by Model', fontsize=12)
axes[1].invert_yaxis()
for i, val in enumerate(results_df['Time (s)']):
    axes[1].text(val + 0.01, i, f'{val:.2f}s', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('comparison_notebook1.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Chart saved as comparison_notebook1.png')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  NOTEBOOK 1 SUMMARY')
print('=' * 60)
print(f'  Dataset   : CO2 Emissions Canada')
print(f'  Instances : 100 (random sample)')
print(f'  CV Folds  : 3')
print(f'  Metric    : RMSE (Root Mean Squared Error)')
print(f'  Algorithms: {len(models)}')
print(f'  Best Model: {results_df.iloc[0]["Model"]}')
print(f'  Best RMSE : {results_df.iloc[0]["RMSE Mean"]} g/km')
print('=' * 60)
print('\n👉 See Notebook 2 for full dataset (7,385 instances) with 10-Fold CV')